In [8]:
import qiskit_metal as qm
from qiskit_metal import designs, draw, Dict
from qiskit_metal.qlibrary.core import QComponent
from shapely.geometry import box, JOIN_STYLE



In [25]:
design = designs.design_planar.DesignPlanar()
design.overwrite_enabled = True   
design._chips.main.size.size_x = '5mm'
design._chips.main.size.size_y = '5mm' 
gui = qm.MetalGUI(design)

02:52PM 35s INFO [_start_renderers]: Renderer=hfss skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
02:52PM 35s INFO [_start_renderers]: Renderer=q3d skipped: runtime dependency not installed (QAnsysRenderer requires pyEPR. Install with: pip install 'quantum-metal[ansys]').
02:52PM 35s INFO [_start_renderers]: Renderer=gmsh skipped: runtime dependency not installed (renderer_gmsh requires gmsh. Install with: pip install 'quantum-metal[mesh]' (or the legacy alias 'quantum-metal[fem]')).
02:52PM 35s INFO [_start_renderers]: Renderer=aedt_q3d skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').
02:52PM 35s INFO [_start_renderers]: Renderer=aedt_hfss skipped: runtime dependency not installed (QPyaedt requires pyaedt (ansys.aedt.core). Install with: pip install 'quantum-metal[ansys]').


In [29]:
class myTransmon(QComponent):
    """Transmon qubit component.
 
    Inherits 'QComponent' class.
 
   
    Default Options:
        * trace_width: 'cpw_width' -- Width of the transmission line attached to the transmon
        * trace_gap: 'cpw_gap' -- Gap of the transmission line
        * lead_length: '25um' -- Length of the transmission line attached to the transmon
        * pad_width: '80um' -- Width of the transmon pad
        * pad_height: '80um' -- Height of the transmon pad
        * pad_gap: '58um' -- Gap of the transmon pad
        * taper_height: '122um' -- Height of the taper from the transmon to the transmission line
    """
 
    default_options = Dict(
        trace_width="25um",
        lead_length="225um",
        pad_side_length="100um",
        pad_gap="58um",
        orientation="90",
    )
 
 
 
 
 
    def make(self):
        """Build the qgeometry for the component."""
        p = self.p
        s = p.pad_side_length
        w = p.trace_width
        w_half = w / 2.0
        lead_length = p.lead_length
        pad_gap = p.pad_gap
 
        #########################################################
        # Pentagon geometry, built with the lead along +x and the pentagon
        # body toward -x. The orientation (default 90) rotates it to point up.
        #
        # The top sides keep the regular-pentagon angle of 36 degrees from the
        # flat top. Their inner ends sit at y = +/- w/2, so the lead fits
        # between them without cutting into them.
        top_angle = np.deg2rad(36.0)
        h1 = s * np.sin(top_angle)                # depth of the shoulders below the top
        y_shoulder = w_half + s * np.cos(top_angle)  # half-width at the shoulders
 
        # The lower sides (length s) run from the shoulders to a base of length s
        dy_lower = y_shoulder - s / 2.0
        h2 = np.sqrt(s**2 - dy_lower**2)          # depth of the lower sides
        H = h1 + h2                               # total pentagon depth
 
        # Pentagon vertices (counterclockwise)
        top_pos = (0, w_half)
        shoulder_pos = (-h1, y_shoulder)
        base_pos = (-H, s / 2.0)
        base_neg = (-H, -s / 2.0)
        shoulder_neg = (-h1, -y_shoulder)
        top_neg = (0, -w_half)
 
        # Main pad: pentagon plus the lead
        transmon = draw.Polygon(
            [
                top_pos,
                shoulder_pos,
                base_pos,
                base_neg,
                shoulder_neg,
                top_neg,
                (lead_length, -w_half),
                (lead_length, w_half),
            ]
        )
 
        # Pocket: the pad grown by pad_gap (sharp corners), trimmed flush with
        # the end of the lead so the readout pin stays open for routing.
        margin = 2 * pad_gap
        pocket = transmon.buffer(pad_gap, join_style=JOIN_STYLE.mitre)
        pocket = pocket.intersection(
            box(-(H + margin), -(y_shoulder + margin), lead_length, y_shoulder + margin)
        )
 
        #########################################################
        # Pin lines. Each side is listed clockwise around the pad, so every pin
        # normal points outward, away from the pad.
        main_pin_line = draw.LineString([(lead_length, w_half), (lead_length, -w_half)])
 
        # Pin names describe positions with the default orientation of 90
        pin_upper_left = draw.LineString([shoulder_pos, top_pos])
        pin_left = draw.LineString([base_pos, shoulder_pos])
        pin_bottom = draw.LineString([base_neg, base_pos])
        pin_right = draw.LineString([shoulder_neg, base_neg])
        pin_upper_right = draw.LineString([top_neg, shoulder_neg])
 
        polys = [
            main_pin_line, transmon, pocket,
            pin_upper_left, pin_left, pin_bottom, pin_right, pin_upper_right,
        ]
 
        # Rotate and translate everything
        polys = draw.rotate(polys, p.orientation, origin=(0, 0))
        polys = draw.translate(polys, xoff=p.pos_x, yoff=p.pos_y)
        [
            main_pin_line, transmon, pocket,
            pin_upper_left, pin_left, pin_bottom, pin_right, pin_upper_right,
        ] = polys
 
        # Add to the qgeometry table
        self.add_qgeometry("poly", dict(transmon=transmon), layer=p.layer)
        self.add_qgeometry("poly", dict(pocket=pocket), layer=p.layer, subtract=True)
 
        # Pins
        self.add_pin("readout", points=list(main_pin_line.coords),
                     width=w, input_as_norm=False)
        self.add_pin("upper_left", points=list(pin_upper_left.coords),
                     width=w, input_as_norm=False)
        self.add_pin("left", points=list(pin_left.coords),
                     width=w, input_as_norm=False)
        self.add_pin("bottom", points=list(pin_bottom.coords),
                     width=w, input_as_norm=False)
        self.add_pin("right", points=list(pin_right.coords),
                     width=w, input_as_norm=False)
        self.add_pin("upper_right", points=list(pin_upper_right.coords),
                     width=w, input_as_norm=False)
 

In [40]:
import numpy as np
from qiskit_metal import draw, Dict
from qiskit_metal.qlibrary.core import QComponent
from airbridge_route import _fillet_centerline

ARC_POINTS = 33   # points per rounded corner (airbridge_route default is 17)


# ----------------------------------------------------------------------------------
# path helpers
# ----------------------------------------------------------------------------------
def _arclength(coords):
    """Cumulative arc length along a polyline."""
    seg = np.linalg.norm(np.diff(coords, axis=0), axis=1)
    return np.concatenate([[0.0], np.cumsum(seg)])


def _s_at_point(coords, pt):
    """Arc-length position (along the polyline) of the point on it closest to `pt`."""
    pt = np.asarray(pt, dtype=float)
    cum = _arclength(coords)
    best_d, best_s = np.inf, 0.0
    for k in range(len(coords) - 1):
        a, ab = coords[k], coords[k + 1] - coords[k]
        L2 = ab @ ab
        t = 0.0 if L2 == 0 else float(np.clip((pt - a) @ ab / L2, 0.0, 1.0))
        d = np.linalg.norm(a + t * ab - pt)
        if d < best_d:
            best_d, best_s = d, cum[k] + t * np.sqrt(L2)
    return best_s


def _check_fillets(points, coords, r, tol=1e-9):
    """Guard the rounding done by _fillet_centerline (any corner angle).

    It leaves a corner sharp, without a word, when its arc does not fit, and it does not
    notice two neighbouring arcs overlapping. A corner turning by theta uses a tangent
    length r*tan(theta/2) on each side, so every segment must be at least as long as the
    tangent lengths of the corners at its two ends."""
    pts = np.asarray(points, dtype=float)
    d = np.diff(pts, axis=0)
    seg = np.linalg.norm(d, axis=1)
    if np.any(seg < tol):
        raise ValueError("Zero-length segment in the waveguide path.")
    u = d / seg[:, None]
    cross = u[:-1, 0] * u[1:, 1] - u[:-1, 1] * u[1:, 0]
    theta = np.abs(np.arctan2(cross, np.einsum("ij,ij->i", u[:-1], u[1:])))
    if np.any(theta < 1e-6):
        raise ValueError("Collinear waypoints in the waveguide path.")
    t = r * np.tan(theta / 2.0)
    need = np.zeros(len(seg))
    need[1:] += t
    need[:-1] += t
    short = np.where(seg < need - tol)[0]
    if len(short):
        i = int(short[0])
        raise ValueError(
            f"Segment {[round(float(v), 5) for v in pts[i]]} -> {[round(float(v), 5) for v in pts[i + 1]]} is "
            f"{seg[i] * 1e3:.1f} um but its fillets need {need[i] * 1e3:.1f} um. Reduce "
            "`fillet` or lengthen coupling_length / lane positions / corridor_drop / "
            "block_b_drop.")
    for v in pts[1:-1]:
        if np.any(np.all(np.abs(coords - v) < tol, axis=1)):
            raise ValueError(f"Corner at {tuple(v)} was left sharp by the fillet.")


def _min_dist_to_segments(P, A, B):
    """For each point in P (N,2): distance to the nearest of the segments A[k] -> B[k]."""
    AB = B - A
    L2 = np.maximum((AB ** 2).sum(1), 1e-30)
    t = np.clip(((P[:, None, :] - A[None]) * AB[None]).sum(2) / L2[None], 0.0, 1.0)
    Q = A[None] + t[..., None] * AB[None]
    return np.sqrt(((P[:, None, :] - Q) ** 2).sum(2)).min(1)


# ----------------------------------------------------------------------------------
# transmon outline and the five coupling edges
# ----------------------------------------------------------------------------------
def _pentagon(s, w):
    """Pentagon vertices of myTransmon, CLOCKWISE, lead pointing up (orientation 90),
    origin at the lead/pad junction: [top_right, shoulder_right, base_right, base_left,
    shoulder_left, top_left]. Same formulas as myTransmon.make()."""
    top = np.deg2rad(36.0)
    h1 = s * np.sin(top)
    y_sh = w / 2.0 + s * np.cos(top)
    h2 = np.sqrt(s ** 2 - (y_sh - s / 2.0) ** 2)
    H = h1 + h2
    return np.array([(w / 2, 0.0), (y_sh, -h1), (s / 2, -H),
                     (-s / 2, -H), (-y_sh, -h1), (-w / 2, 0.0)])


def _edge_frames(V, c, L):
    """For the five pad edges (clockwise: upper_right, right, bottom, left, upper_left):
    unit tangent t (direction of travel), outward unit normal n, the centre P of the
    coupling run (edge midpoint pushed out by c) and its two ends tip_a -> tip_b."""
    frames = []
    for i in range(5):
        a, b = V[i], V[i + 1]
        t = (b - a) / np.linalg.norm(b - a)
        n = np.array([-t[1], t[0]])                   # outward for a clockwise polygon
        P = (a + b) / 2.0 + c * n
        frames.append(dict(t=t, n=n, P=P, tip_a=P - L / 2 * t, tip_b=P + L / 2 * t))
    return frames


def _y_on_line(p0, n, x):
    """y of the line through p0 with direction n at abscissa x."""
    return p0[1] + (x - p0[0]) / n[0] * n[1]


def _rows(y_first, n, pitch, x_left, x_right, x_final):
    """Vertices for n horizontal rows stacked downward from y_first.

    Row 0 heads right from the vertex the caller has already added; rows alternate.
    n must be even so the last row heads left, ending at x_final."""
    v = []
    for k in range(n):
        y = y_first - k * pitch
        x = x_right if k % 2 == 0 else (x_left if k < n - 1 else x_final)
        v.append((x, y))
        if k < n - 1:
            v.append((x, y - pitch))
    return v


def _layout(g, nA, WA, nB, WB):
    """Full waypoint list plus the five coupling-run centres.

    Right half (chip right edge -> bottom coupling run):
      in along the top -> down lane -> leg to contact 1 (upper-right edge) -> run along the
      edge -> leg and lane back up -> block A rows -> corridor down -> leg to contact 2
      (right edge) -> run along the edge -> leg and corridor down -> block B rows
      -> up under the qubit -> contact 3 (bottom edge, shared with the left half).
    The left half is the mirror image about x = qubit_x, traversed in reverse.
    Every leg runs along the edge's outward normal; every coupling run is parallel to its edge."""
    pitch = g["pitch"]
    e1, e2, e3 = g["e"][0], g["e"][1], g["e"][2]
    x_cor, x_dn, x_up, x_min = g["x_cor"], g["x_dn"], g["x_up"], g["x_min"]

    ya2 = _y_on_line(e2["tip_a"], e2["n"], x_cor)         # corridor meets contact 2's legs
    yb2 = _y_on_line(e2["tip_b"], e2["n"], x_cor)
    ya1 = _y_on_line(e1["tip_a"], e1["n"], x_dn)          # lanes meet contact 1's legs
    yb1 = _y_on_line(e1["tip_b"], e1["n"], x_up)

    y_lastA = ya2 + g["corridor_drop"]
    y1 = y_lastA + (nA - 1) * pitch                       # first row of block A
    Y = y1 + pitch                                        # incoming run, one pitch above it
    yB0 = yb2 - g["block_b_drop"]                         # first row of block B

    R = [(g["start_x"], Y), (x_dn, Y), (x_dn, ya1), tuple(e1["tip_a"]), tuple(e1["tip_b"]),
         (x_up, yb1), (x_up, y1)]
    R += _rows(y1, nA, pitch, x_min, x_min + WA, x_cor)
    R += [(x_cor, ya2), tuple(e2["tip_a"]), tuple(e2["tip_b"]), (x_cor, yb2), (x_cor, yB0)]
    R += _rows(yB0, nB, pitch, x_min, x_min + WB, e3["tip_a"][0])
    R += [tuple(e3["tip_a"])]

    qx = g["qx"]
    M = [(2 * qx - x, y) for x, y in reversed(R)]
    mir = lambda P: (2 * qx - P[0], P[1])
    if g["open_left"]:
        # Drop the exit leg, the top lane and the run to the chip's left edge: the line is
        # cut off inside contact 5's run, `open_end_overshoot` past that run's centre.
        M = M[:-4]
        u = np.array(mir(e1["tip_a"])) - np.array(M[-1])         # along the run, toward the lead
        M.append(tuple(np.array(mir(e1["P"])) + g["ovs"] * u / np.linalg.norm(u)))
    centres = [tuple(e1["P"]), tuple(e2["P"]), tuple(e3["P"]), mir(e2["P"]), mir(e1["P"])]
    return R + M, centres


class GiantAtomWaveguide(QComponent):
    """Giant-atom waveguide coupled to the pentagon transmon at five points.

    Coupling order (global frame, qubit lead pointing up, i.e. myTransmon orientation 90):

        1 upper_right -> 2 right -> 3 bottom -> 4 left -> 5 upper_left

    At each point the waveguide makes a hairpin: it approaches along the edge's outward
    normal, runs STRAIGHT AND PARALLEL to the edge for `coupling_length`, `coupling_gap`
    away from the pad (edge to trace edge), and leaves along the normal again. It never
    enters the transmon; make() raises if any part of the trace gets closer than
    trace_width/2 + trace_gap to the pad outline (that would etch into the pad).

    Between consecutive coupling points the waveguide runs `target_length` (run centre to
    run centre), taken up by serpentines of horizontal rows. It enters at the chip's right
    edge above the qubit. With `left_end_open` (default) it ends in an OPEN CIRCUIT inside the
    last coupling run (contact 5, upper_left): the centre conductor stops there and the ground
    cut-out is capped `open_end_gap` beyond it. Otherwise it leaves through the chip's left edge.
    The layout is mirror-symmetric about x = qubit_x, so the qubit must sit on the chip's centre line.
    """

    default_options = Dict(
        # --- qubit geometry (match myTransmon options; orientation 90 => lead points up)
        qubit_x="0mm",            # qubit pos_x (also the mirror axis)
        qubit_y="0mm",            # qubit pos_y (lead / pad junction)
        qubit_pad_side="100um",   # myTransmon pad_side_length
        qubit_lead_width="25um",  # myTransmon trace_width
        qubit_lead_length="225um",  # myTransmon lead_length
        # --- coupling runs (parallel to each pad edge)
        coupling_gap="10um",      # pad edge -> trace edge (must be >= trace_gap)
        coupling_length="100um",  # length of the hairpin's run along the edge (>= 2*fillet;
                                  # the part that is truly straight is this minus 2*fillet)
        # --- routing around the qubit
        lane_in_x="70um",         # vertical lane that brings the waveguide down to contact 1
        lane_out_x="130um",       # vertical lane that carries it back up to block A
        corridor_x="200um",       # vertical lane linking block A, contact 2 and block B
        corridor_drop="120um",    # block A's last row sits this far above contact 2's upper leg
        block_b_drop="120um",     # block B's first row sits this far below contact 2's lower leg
        # --- left end of the waveguide
        left_end_open="True",       # True: open circuit inside contact 5's run; False: out to the chip edge
        open_end_overshoot="20um",  # how far the open end continues past the centre of that run
        open_end_gap="6um",         # ground gap beyond the open end (etch cap)
        # --- serpentines
        target_length="22mm",     # coupling-run centre -> next coupling-run centre
        serp_x_min="0.25mm",      # left turns of the serpentine rows
        serp_x_max="2.3mm",       # right-most x the serpentine rows may reach
        pitch="60um",             # row-to-row spacing (must be >= 2*fillet)
        fillet="30um",
        # --- chip / layout
        start_x="2.5mm",          # x of the chip's right edge (the left edge is its mirror)
        trace_width="10um",
        trace_gap="6um",
        layer="5",
        chip="main",
    )
    component_metadata = Dict(short_name="waveguide", _qgeometry_table_path="True")

    def get_points(self):
        return self._waveguide_points

    def make(self):
        p = self.p
        r, w, pitch, gap = p.fillet, p.trace_width, p.pitch, p.trace_gap
        qx, qy = p.qubit_x, p.qubit_y
        
        # --- transmon outline and the five parallel coupling runs
        V = _pentagon(p.qubit_pad_side, p.qubit_lead_width) + np.array([qx, qy])
        e = _edge_frames(V, p.coupling_gap + w / 2.0, p.coupling_length)
        e1, e2 = e[0], e[1]

        x_dn, x_up, x_cor = qx + p.lane_in_x, qx + p.lane_out_x, qx + p.corridor_x


        open_left = str(p.left_end_open).lower() in ("true", "1", "yes")
        
        g = dict(open_left=open_left, ovs=p.open_end_overshoot, qx=qx, start_x=p.start_x, pitch=pitch, e=e, x_dn=x_dn, x_up=x_up,
                 x_cor=x_cor, x_min=qx + p.serp_x_min,
                 corridor_drop=p.corridor_drop, block_b_drop=p.block_b_drop)

        # --- size the two serpentine blocks so each link is exactly target_length.
        # Link length is linear in the row width W with slope n (n = rows), so for each
        # even n we probe at W = 1 mm and solve W in closed form; take the smallest n that
        # fits under serp_x_max.
        W_max = p.serp_x_max - p.serp_x_min

        def link_length(nA, WA, nB, WB, i):
            pts, cen = _layout(g, nA, WA, nB, WB)
            coords = _fillet_centerline(pts, r, ARC_POINTS)
            s = [_s_at_point(coords, c) for c in cen[i:i + 2]]
            return s[1] - s[0]

        def solve(i, nA_fixed=None, WA_fixed=None):
            for n in range(2, 400, 2):
                args = (n, 1.0, 2, 1.0) if i == 0 else (nA_fixed, WA_fixed, n, 1.0)
                W = 1.0 + (p.target_length - link_length(*args, i)) / n
                if W <= W_max:
                    break
            if W < 2 * r:
                raise ValueError("Serpentine rows would be shorter than 2*fillet; "
                                 "increase target_length or reduce serp_x_max.")
            return n, W

        nA, WA = solve(0)
        nB, WB = solve(1, nA, WA)

        points, centres = _layout(g, nA, WA, nB, WB)
        coords = _fillet_centerline(points, r, ARC_POINTS)
        _check_fillets(points, coords, r)
        cum = _arclength(coords)
        s_c = [_s_at_point(coords, c) for c in centres]

        # --- the trace must stay out of the transmon (pad outline + readout lead)
        lead_top = qy + p.qubit_lead_length
        A = np.vstack([V[:-1], V[0], V[5], [V[0][0], lead_top]])
        B = np.vstack([V[1:], [V[0][0], lead_top], [V[5][0], lead_top], [V[5][0], lead_top]])
        d_pad = _min_dist_to_segments(coords, A, B)


        # bookkeeping 
        self._waveguide_points = np.array(points, dtype=float)
        self.coupling_centres = [tuple(c) for c in centres]
        self.coupling_spacings = list(np.diff(s_c))     # run 1->2, 2->3, 3->4, 4->5
        self.rows_block_a, self.row_width_block_a = nA, WA
        self.rows_block_b, self.row_width_block_b = nB, WB
        self.length_total = cum[-1]
        self.min_gap_to_transmon = d_pad.min() - w / 2  # trace edge to pad / lead metal

        centerline = draw.LineString(coords)
        etch_coords = coords
        if open_left:
            # cap the open end: the etch continues open_end_gap past the end of the trace
            u_end = coords[-1] - coords[-2]
            u_end = u_end / np.linalg.norm(u_end)
            etch_coords = np.vstack([coords, coords[-1] + p.open_end_gap * u_end])
            self.open_end = tuple(coords[-1])
        etch_line = draw.LineString(etch_coords)
        self.add_qgeometry("path", {"waveguide": centerline},
                           width=w, fillet=0, layer=p.layer, chip=p.chip)
        self.add_qgeometry("path", {"waveguide_etch": etch_line},
                           width=w + 2 * gap, fillet=0,
                           layer=p.layer, subtract=True, chip=p.chip)

        # pins: outward normal = direction of travel at each end
        def pin(name, end, dirn):
            t = np.array([dirn[1], -dirn[0]])                # tangent so that normal == dirn
            self.add_pin(name, input_as_norm=False, width=w,
                         points=np.array([end - t * w / 2, end + t * w / 2]))

        pin("waveguide_connect", np.array(points[0], float), (1, 0))
        if not open_left:                                    # an open end has nothing to connect to
            d_end = np.sign(points[-1][0] - points[-2][0])   # -1: leaves through the left edge
            pin("waveguide_end", np.array(points[-1], float), (d_end, 0))

In [41]:
design.template_options.pop(myTransmon._get_unique_class_name(), None)
design.template_options.pop(GiantAtomWaveguide._get_unique_class_name(), None)

q1 = myTransmon(design, 'Q1', options=dict(pos_x='0um', pos_y='0um'))



wg = GiantAtomWaveguide(design=design, name="waveguide")



gui.rebuild()
gui.autoscale()

print("coupling spacings (mm):", [f"{s:.4f}" for s in wg.coupling_spacings])
print(f"rows: block A {wg.rows_block_a}, block B {wg.rows_block_b}; total length {wg.length_total:.2f} mm")


coupling spacings (mm): ['22.0000', '22.0000', '22.0000', '22.0000']
rows: block A 12, block B 10; total length 91.21 mm
